# سبق 09 - مابعدِ ادراک کا ڈیزائن نمونہ


## سیٹ اپ

یہ نوٹ بک Microsoft Agent Framework استعمال کرتے ہوئے Metacognition ڈیزائن پیٹرن کی مثال پیش کرتی ہے۔

**ضروریات:**
- Azure OpenAI کی تعیناتی جو ماحولیاتی متغیرات کے ذریعے ترتیب دی گئی ہو
- Azure CLI تصدیق شدہ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## مابعد ادراک کیا ہے؟

مابعد ادراک **سوچ کے بارے میں سوچ** ہے۔ 

مصنوعی ذہانت کے ایجنٹس کے سیاق و سباق میں، اس کا مطلب ایسے ایجنٹس بنانا ہے جو کر سکیں:

- **خود انعکاس** اپنے ہی نتائج اور استدلالی عمل پر
- **غلطیوں کا پتہ لگائیں** اور خاموشی سے ناکام ہونے کے بجائے بخوبی بحال ہوں
- **جانچیں** کہ آیا ان کے جوابات مکمل اور مددگار ہیں
- **اپنی حکمتِ عملی ڈھالیں** جب ایک ابتدائی طریقہ کار کام نہ کرے (مثلاً، بیک اپ سسٹم پر واپس جانا)

ایک مابعد ادراکی ایجنٹ صرف سوالات کا جواب نہیں دیتا — یہ اپنی کارکردگی کی نگرانی کرتا ہے اور فوری طور پر حالات کے مطابق ایڈجسٹ کرتا ہے۔


## پرائمری اور بیک اپ اوزار

ایک عام میٹاکگنیشن پیٹرن **بیک اپ حکمتِ عملی** ہے۔

ایجنٹ پہلے ایک پرائمری اوزار آزمانے کی کوشش کرتا ہے؛ اگر وہ ناکام ہو جائے (مثلاً، 404 ایرر)، تو ایجنٹ ناکامی کو پہچان لیتا ہے اور شفافیت کے ساتھ بیک اپ اوزار پر منتقل ہو جاتا ہے۔

یہ حقیقی دنیا کے نظام کی عکاسی کرتا ہے جہاں پرائمری خدمات دستیاب نہ بھی ہوں اور ایجنٹس کو متبادل راستہ منتخب کرنے سے پہلے خود مسئلے کی تشخیص کرنی پڑتی ہے۔

نیچے ہم دو پرواز تلاش کرنے والے اوزار متعین کرتے ہیں:
- **پرائمری** — اس میں پیرس، ٹوکیو، اور بارسلونا شامل ہیں
- **بیک اپ** — اس میں برلن، سڈنی، اور نیو یارک سٹی شامل ہیں


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## خود عکاس ایجنٹ برائے خرابی کی بازیابی

ذیل میں دیا گیا ایجنٹ ہدایت یافتہ ہے کہ وہ پہلے بنیادی پرواز کے نظام کو آزمانا شروع کرے، ناکامیوں کو پہچانے، اور شفاف طور پر بیک اپ نظام پر واپس چلا جائے۔ ہر جواب کے بعد وہ مختصراً خود کا جائزہ لیتا ہے کہ آیا اس نے صارف کے سوال کا مکمل جواب دیا تھا یا نہیں۔


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## خود تشخیصی پیٹرن

مابعد ادراک کا ایک اور پہلو **خود-تشخیص** ہے: ایک الگ ایجنٹ (یا وہی ایجنٹ دوسری بار) کسی جواب کا مکمل ہونے، درستگی، اور مفیدیت کے لحاظ سے جائزہ لیتا ہے۔

ذیل میں ہم ایک `ResponseEvaluator` ایجنٹ بناتے ہیں جو سفری ایجنٹ کے جوابات کو تین ابعاد پر اسکور کرتا ہے۔


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ Microsoft Agent Framework استعمال کرتے ہوئے **متافکری ایجنٹس** کیسے بنائے جاتے ہیں:

- **خود عکاسی**: ایسے ایجنٹس جو اپنے استدلال کی نگرانی کرتے ہیں اور شفاف طریقے سے بتاتے ہیں کہ کیا ہوا۔
- **فیل بیک کے ساتھ خرابی سے بازیابی**: ایک پرائمری + بیک اپ ٹول پیٹرن جہاں ایجنٹ ناکامیوں (مثلاً 404 غلطیاں) کا پتہ لگا کر خودکار طور پر متبادل ماخذ آزما لیتا ہے۔
- **خود تشخیص**: ایک الگ جانچنے والا ایجنٹ جو جوابات کو مکملیت، درستگی، اور مفیدیت کے لحاظ سے اسکور کرتا ہے۔

یہ پیٹرن ایجنٹس کو زیادہ مضبوط، شفاف، اور قابلِ اعتماد بناتے ہیں — پیداواری تعیناتیوں کے لیے اہم خصوصیات۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
دستبرداری:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہِ کرم نوٹ کریں کہ خودکار تراجم میں غلطیاں یا عدم مطابقت ہو سکتی ہے۔ اصل دستاویز کو اس کی مادری زبان میں ہی مستند ماخذ سمجھا جانا چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی تجویز دی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
